# Setup

In [6]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd

# helper functions
from tiktok_api_helpers import *

## Settings

In [14]:
new_access_token = False
use_refresh_token = False

## Constants

In [8]:
base_url = "https://open-api.tiktokglobalshop.com"

In [9]:
load_dotenv()  # reads .env in the current directory into environment variables

app_key = os.environ.get("TIKTOK_APP_KEY")
app_secret = os.environ.get("TIKTOK_APP_SECRET")
auth_code = os.environ.get("TIKTOK_AUTH_CODE")
shop_cipher = os.environ.get("SHOP_CIPHER")

In [ ]:
load_dotenv()  # reads .env in the current directory into environment variables

app_key = os.environ.get("TIKTOK_APP_KEY")
app_secret = os.environ.get("TIKTOK_APP_SECRET")
access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
shop_cipher = os.environ.get("SHOP_CIPHER")

base_url = "https://open-api.tiktokglobalshop.com"
path = "/affiliate_seller/202508/marketplace_creators/search"

INPUT_CSV = "all_creators_handleonly.csv"
CONSOLIDATED_CSV = "creators_found.csv"
MANIFEST_CSV = "creators_manifest.csv"
DELAY_BETWEEN_CALLS = 1.0  # seconds, to be gentle on the rate limit
RATE_LIMIT_CODE = 36009002

## Load data

In [48]:
list_all_creators = pd.read_csv("all_creators_handleonly.csv")['Handle'].tolist()

## Helper functions

# API Setup

## 1. Get Access Token

In [15]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

## 2. Get Shop Cipher

(True, 'SHOP_CIPHER', 'ROW_N6cpKQAAAADl57_xyhcxuGLS8aCEeYMb')

# Step 1: Get Creator Open IDs

In [ ]:
list_all_creators = pd.read_csv(INPUT_CSV)["Handle"].tolist()

# Load or initialize the manifest (tracks which handles are already found)
if Path(MANIFEST_CSV).exists():
    manifest = pd.read_csv(MANIFEST_CSV)
else:
    manifest = pd.DataFrame({"handle": list_all_creators, "found": False})

# Load or initialize consolidated creator data pulled so far
if Path(CONSOLIDATED_CSV).exists():
    df_creators = pd.read_csv(CONSOLIDATED_CSV)
else:
    df_creators = pd.DataFrame()

handles_to_find = manifest.loc[~manifest["found"], "handle"].tolist()
print(f"{len(handles_to_find)} handles left to find out of {len(manifest)} total.\n")

# Phase 1: chunk_size=5, single pass through everything not yet found
still_not_found, df_creators = run_pass(handles_to_find, chunk_size=5, df_creators=df_creators)

found_usernames = set(df_creators["username"].str.lower())
manifest["found"] = manifest["handle"].str.lstrip("@").str.lower().isin(found_usernames)
manifest.to_csv(MANIFEST_CSV, index=False)
print(f"\nPhase 1 done. {len(still_not_found)} handles still not found.\n")

# Phase 2: chunk_size=1, only for leftovers from phase 1
if still_not_found:
    _, df_creators = run_pass(still_not_found, chunk_size=1, df_creators=df_creators)

    found_usernames = set(df_creators["username"].str.lower())
    manifest["found"] = manifest["handle"].str.lstrip("@").str.lower().isin(found_usernames)
    manifest.to_csv(MANIFEST_CSV, index=False)

print(f"\nDone. {int(manifest['found'].sum())} / {len(manifest)} handles found.")
print(f"Consolidated data: {Path(CONSOLIDATED_CSV).resolve()}")
print(f"Manifest: {Path(MANIFEST_CSV).resolve()}")

In [73]:
keyword_list = list_all_creators[5:10]

In [74]:
keyword = '@'+"|@".join(keyword_list)
print(keyword)

@iphonatics|@moto.math|@annalmoite|@zy.mendoza.gerond|@mikebalolong


In [86]:
path = "/affiliate_seller/202508/marketplace_creators/search"

# setup parameters
params = {
    "app_key": app_key,
    "timestamp": int(time.time()),
    "shop_cipher": shop_cipher,
    "page_size":20,
}

# setup search keywords
body_dict = {
    "keyword": keyword,
    'search_key': search_key
}
body = json.dumps(body_dict)
signed_params = build_signed_params(path, params, app_secret, body)

# send request
response = requests.post(
    f"{base_url}{path}",
    params=signed_params,
    data=body,
    headers={"x-tts-access-token": access_token, "content-type": "application/json"},
    timeout=15,
)
print(f"Status: {response.status_code}")

Status: 200


In [87]:
search_key = response.json()['data']['search_key']

In [81]:
keyword = '@iphonatics|@annalmoite|@zy.mendoza.gerond'

In [88]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv,gmv_range,live_gmv,nickname,selection_region,top_follower_demographics,username,video_gmv
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2874,"[824328, 601152, 603014]",7uArVgAAAACOV4AEeERJl-yievH_HBssyCoWST9L0ZrCkD...,194943,"{'amount': '267947.693812', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...","{'amount': '2.658677', 'currency': 'USD'}",Miss Ann ✿,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",annalmoite,"{'amount': '267737.248135', 'currency': 'USD'}"
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2157,"[601739, 600001]",yGLEYQAAAACOV4AEeERJl-yievH_HBsseWu9pSsIf9zLL8...,611913,"{'amount': '211924.621037', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...","{'amount': '6.83353', 'currency': 'USD'}",IPHONATICS,PH,"{'age_ranges': ['AGE_RANGE_18_24', 'AGE_RANGE_...",iphonatics,"{'amount': '208758.776693', 'currency': 'USD'}"
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601739, 604453, 600154]",S7jHnQAAAACOV4AEeERJl-yievH_HBsspmDYP-pIAVrUlv...,31731,NaN,{'formatted_range': '₱10K+'},NaN,Almira Carriaga,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",almiracarriaga,NaN
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",9Ovn_wAAAACOV4AEeERJl-yievH_HBssS7BIL5paEZ6qYL...,531,NaN,{'formatted_range': '₱10K+'},NaN,Maria,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",user5844826884387,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",3MxrrwAAAACOV4AEeERJl-yievH_HBssPMIZu50b8vQOzm...,13,"{'amount': '5825500.518508', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",NaN,iflyad03,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",iflyad03,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 605196]",T51HRwAAAACOV4AEeERJl-yievH_HBsscpim0sGFIiscz9...,1,NaN,{'formatted_range': '₱10K+'},NaN,ShopAtHomePH,PH,"{'age_ranges': ['AGE_RANGE_35_44'], 'major_gen...",shopathomeph,NaN
6,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601739, 605196, 601450]",Bk01NQAAAACOV4AEeERJl-yievH_HBssjYBT2NCiw6gm5a...,785,"{'amount': '28265.648061', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",NaN,Nicolas Pagquil,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",nicolas.pagquil,NaN
7,{'url': 'https://p16-common-sign.tiktokcdn.com...,102597,0,"[601739, 601755]",jSATJgAAAACOV4AEeERJl-yievH_HBssugnh2geIdZYbIX...,114461,NaN,{'formatted_range': '₱10K+'},NaN,Hari ng Chamba,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",haringchamba,NaN
8,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,281,"[601739, 604968, 605196]",4WFi3gAAAACOV4AEeERJl-yievH_HBss5966XF9PQoqdpS...,20709,"{'amount': '22730.431964', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",NaN,Egi finds,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",egii_tv_06,"{'amount': '22730.431964', 'currency': 'USD'}"
9,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",yJHLEAAAAACOV4AEeERJl-yievH_HBss0zwVS1hGagjS2u...,0,NaN,{'formatted_range': '₱10K+'},NaN,Manh Vinh,PH,"{'age_ranges': [], 'major_gender': {'gender': ...",manh.vinh82,NaN


In [80]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,live_gmv,video_gmv
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,1230,"[601450, 601739, 601755]",dbY-eAAAAACOV4AEeERJl-yievH_HBss139WeLpZGcLm-7...,795812,{'formatted_range': '₱10K+'},Mike Balolong,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",mikebalolong,NaN,NaN,NaN
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,2093,4506,"[605196, 603014, 604453]",WQB7JgAAAACOV4AEeERJl-yievH_HBssNH0R3AlIg46elZ...,53365,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Moto Math,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",moto.math,"{'amount': '243375.06568', 'currency': 'USD'}","{'amount': '485.304058', 'currency': 'USD'}","{'amount': '242618.434792', 'currency': 'USD'}"
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601739, 604453, 600154]",S7jHnQAAAACOV4AEeERJl-yievH_HBsspmDYP-pIAVrUlv...,31731,{'formatted_range': '₱10K+'},Almira Carriaga,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",almiracarriaga,NaN,NaN,NaN
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",9Ovn_wAAAACOV4AEeERJl-yievH_HBssS7BIL5paEZ6qYL...,531,{'formatted_range': '₱10K+'},Maria,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",user5844826884387,NaN,NaN,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",3MxrrwAAAACOV4AEeERJl-yievH_HBssPMIZu50b8vQOzm...,13,"{'currency': 'USD', 'formatted_range': '₱10K+'...",iflyad03,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",iflyad03,"{'amount': '5825500.518508', 'currency': 'USD'}",NaN,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 605196]",T51HRwAAAACOV4AEeERJl-yievH_HBsscpim0sGFIiscz9...,1,{'formatted_range': '₱10K+'},ShopAtHomePH,PH,"{'age_ranges': ['AGE_RANGE_35_44'], 'major_gen...",shopathomeph,NaN,NaN,NaN
6,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601739, 605196, 601450]",Bk01NQAAAACOV4AEeERJl-yievH_HBssjYBT2NCiw6gm5a...,785,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Nicolas Pagquil,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",nicolas.pagquil,"{'amount': '28265.648061', 'currency': 'USD'}",NaN,NaN
7,{'url': 'https://p16-common-sign.tiktokcdn.com...,102597,0,"[601739, 601755]",jSATJgAAAACOV4AEeERJl-yievH_HBssugnh2geIdZYbIX...,114461,{'formatted_range': '₱10K+'},Hari ng Chamba,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",haringchamba,NaN,NaN,NaN
8,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,281,"[601739, 604968, 605196]",4WFi3gAAAACOV4AEeERJl-yievH_HBss5966XF9PQoqdpS...,20709,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Egi finds,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",egii_tv_06,"{'amount': '22730.431964', 'currency': 'USD'}",NaN,"{'amount': '22730.431964', 'currency': 'USD'}"
9,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",yJHLEAAAAACOV4AEeERJl-yievH_HBss0zwVS1hGagjS2u...,0,{'formatted_range': '₱10K+'},Manh Vinh,PH,"{'age_ranges': [], 'major_gender': {'gender': ...",manh.vinh82,NaN,NaN,NaN


In [69]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,video_gmv
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601352, 601152]",I2IWFwAAAACOV4AEeERJl-yievH_HBssOVaC2_feBRCR7I...,894,{'formatted_range': '₱100-₱1K'},Jpongs_,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",jpolimar2nd,NaN,NaN
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601152, 600942, 603014]",FJXiWgAAAACOV4AEeERJl-yievH_HBss6kZoZmHqG7TSuA...,1922,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_2000,NaN,NaN
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,3921,"[600001, 700437, 601739]",0Ktl2wAAAACOV4AEeERJl-yievH_HBss98ZZMcc5NxLGOJ...,127228,"{'currency': 'USD', 'formatted_range': '₱10K+'...",david sy,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",david.sy25,"{'amount': '198894.218417', 'currency': 'USD'}","{'amount': '198810.840923', 'currency': 'USD'}"
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[605196, 603014, 604579]",ekTRtQAAAACOV4AEeERJl-yievH_HBssjAnMhoUO18D0YG...,7043,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jomaryeedump52,NaN,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2304,"[601352, 603014, 604453]",Y5EkgwAAAACOV4AEeERJl-yievH_HBssTrKMtvBlwwqFdF...,88875,{'formatted_range': '₱10K+'},Jp Olimar🇵🇭,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jpongs_,NaN,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,1431,"[600942, 801928, 602284]",OjX16gAAAACOV4AEeERJl-yievH_HBssLMogjZiSaMeXxo...,27361,{'formatted_range': '₱10K+'},Karen Anonas,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",karenanonas,NaN,NaN


In [63]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,video_gmv,live_gmv
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601352, 601152]",I2IWFwAAAACOV4AEeERJl-yievH_HBssOVaC2_feBRCR7I...,894,{'formatted_range': '₱100-₱1K'},Jpongs_,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",jpolimar2nd,NaN,NaN,NaN
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 700645, 600942]",4-8EuQAAAACOV4AEeERJl-yievH_HBssVSbe7_5LcPqJXc...,2842,{'formatted_range': '₱100-₱1K'},Dump ni jomar yee,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_5,NaN,NaN,NaN
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601152, 600942, 603014]",FJXiWgAAAACOV4AEeERJl-yievH_HBss6kZoZmHqG7TSuA...,1922,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_2000,NaN,NaN,NaN
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[824584, 601152, 601450]",L7PcYwAAAACOV4AEeERJl-yievH_HBsshmh4OB3xdmSeP7...,1004,{'formatted_range': '₱0-₱100'},yuni okayde😊,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedumpz,NaN,NaN,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[824584, 600001, 600024]",giMQhgAAAACOV4AEeERJl-yievH_HBssdsfXKMGxp82a0j...,169,{'formatted_range': '₱0-₱100'},WenceslaoShop,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump8,NaN,NaN,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,3921,"[600001, 700437, 601739]",0Ktl2wAAAACOV4AEeERJl-yievH_HBss98ZZMcc5NxLGOJ...,127228,"{'currency': 'USD', 'formatted_range': '₱10K+'...",david sy,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",david.sy25,"{'amount': '198894.218417', 'currency': 'USD'}","{'amount': '198810.840923', 'currency': 'USD'}",NaN
6,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[605196, 603014, 604579]",ekTRtQAAAACOV4AEeERJl-yievH_HBssjAnMhoUO18D0YG...,7043,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jomaryeedump52,NaN,NaN,NaN
7,{'url': 'https://p19-common-sign.tiktokcdn.com...,0,1940,"[601352, 824328, 601152]",OlDpxwAAAACOV4AEeERJl-yievH_HBssyJrMiLFJf1B0bD...,26566,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Fernando's Finds,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",majuuuuuuri,"{'amount': '105103.947971', 'currency': 'USD'}","{'amount': '103030.857739', 'currency': 'USD'}",NaN
8,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,43533,"[601450, 601152, 600001]",6vRP6QAAAACOV4AEeERJl-yievH_HBssm6EICBQi-rrSkp...,1566047,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Queen Pitik QP,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",zy.mendoza.gerond,"{'amount': '190698.792037', 'currency': 'USD'}","{'amount': '187435.971618', 'currency': 'USD'}","{'amount': '2.412918', 'currency': 'USD'}"
9,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2304,"[601352, 603014, 604453]",Y5EkgwAAAACOV4AEeERJl-yievH_HBssTrKMtvBlwwqFdF...,88875,{'formatted_range': '₱10K+'},Jp Olimar🇵🇭,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jpongs_,NaN,NaN,NaN


In [39]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,video_gmv
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601352, 601152]",I2IWFwAAAACOV4AEeERJl-yievH_HBssOVaC2_feBRCR7I...,894,{'formatted_range': '₱100-₱1K'},Jpongs_,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",jpolimar2nd,NaN,NaN
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 700645, 600942]",4-8EuQAAAACOV4AEeERJl-yievH_HBssVSbe7_5LcPqJXc...,2842,{'formatted_range': '₱100-₱1K'},Dump ni jomar yee,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_5,NaN,NaN
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601152, 600942, 603014]",FJXiWgAAAACOV4AEeERJl-yievH_HBss6kZoZmHqG7TSuA...,1922,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_2000,NaN,NaN
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[824584, 601152, 601450]",L7PcYwAAAACOV4AEeERJl-yievH_HBsshmh4OB3xdmSeP7...,1004,{'formatted_range': '₱0-₱100'},yuni okayde😊,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedumpz,NaN,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[824584, 600001, 600024]",giMQhgAAAACOV4AEeERJl-yievH_HBssdsfXKMGxp82a0j...,169,{'formatted_range': '₱0-₱100'},WenceslaoShop,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump8,NaN,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601755, 700645, 601739]",XTl2zwAAAACOV4AEeERJl-yievH_HBssRnaxqwIqDw63zr...,7,{'formatted_range': '₱0-₱100'},JOMARYEEDUMP,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",_lunaaserene,NaN,NaN
6,{'url': 'https://p19-common-sign.tiktokcdn.com...,0,3921,"[600001, 700437, 601739]",0Ktl2wAAAACOV4AEeERJl-yievH_HBss98ZZMcc5NxLGOJ...,127228,"{'currency': 'USD', 'formatted_range': '₱10K+'...",david sy,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",david.sy25,"{'amount': '198894.218417', 'currency': 'USD'}","{'amount': '198810.840923', 'currency': 'USD'}"
7,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[605196, 603014, 604579]",ekTRtQAAAACOV4AEeERJl-yievH_HBssjAnMhoUO18D0YG...,7043,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jomaryeedump52,NaN,NaN
8,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,1940,"[601352, 824328, 601152]",OlDpxwAAAACOV4AEeERJl-yievH_HBssyJrMiLFJf1B0bD...,26566,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Fernando's Finds,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",majuuuuuuri,"{'amount': '105103.947971', 'currency': 'USD'}","{'amount': '103030.857739', 'currency': 'USD'}"
9,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2304,"[601352, 603014, 604453]",Y5EkgwAAAACOV4AEeERJl-yievH_HBssTrKMtvBlwwqFdF...,88875,{'formatted_range': '₱10K+'},Jp Olimar🇵🇭,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jpongs_,NaN,NaN


In [28]:
response.json()

{'code': 0,
 'data': {'creators': [],
  'next_page_token': '',
  'search_key': '3sf9V5goXkn1VA8nI9vFWisff4VZFPeFRkX2LsNEFho='},
 'message': 'Success',
 'request_id': '2026071311062493A16B16BDBC1D9DE682'}

In [113]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,live_gmv,video_gmv
0,{'url': 'https://p19-common-sign.tiktokcdn.com...,0,0,"[601352, 601152]",I2IWFwAAAACOV4AEeERJl-yievH_HBssOVaC2_feBRCR7I...,895,{'formatted_range': '₱100-₱1K'},Jpongs_,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",jpolimar2nd,NaN,NaN,NaN
1,{'url': 'https://p19-common-sign.tiktokcdn.com...,0,0,"[601450, 801928, 600154]",4-8EuQAAAACOV4AEeERJl-yievH_HBssVSbe7_5LcPqJXc...,2843,{'formatted_range': '₱100-₱1K'},Dump ni jomar yee,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_5,NaN,NaN,NaN
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[700645, 601739, 602284]",FJXiWgAAAACOV4AEeERJl-yievH_HBss6kZoZmHqG7TSuA...,1923,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_2000,NaN,NaN,NaN
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[602118, 824584, 601739]",L7PcYwAAAACOV4AEeERJl-yievH_HBsshmh4OB3xdmSeP7...,1004,{'formatted_range': '₱0-₱100'},yuni okayde😊,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedumpz,NaN,NaN,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601739, 601755, 700645]",XTl2zwAAAACOV4AEeERJl-yievH_HBssRnaxqwIqDw63zr...,7,{'formatted_range': '₱0-₱100'},JOMARYEEDUMP,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",_lunaaserene,NaN,NaN,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,1541,2090,"[601450, 700437, 602118]",tO3PiwAAAACOV4AEeERJl-yievH_HBssheFI3rImSFcsoj...,30423,{'formatted_range': '₱10K+'},Ms.Arianne,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",cutyywee,NaN,NaN,NaN
6,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601352, 604968, 605248]",ekTRtQAAAACOV4AEeERJl-yievH_HBssjAnMhoUO18D0YG...,7046,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jomaryeedump52,NaN,NaN,NaN
7,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,40710,"[601450, 601152, 600001]",6vRP6QAAAACOV4AEeERJl-yievH_HBssm6EICBQi-rrSkp...,1562234,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Queen Pitik QP,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",zy.mendoza.gerond,"{'amount': '188717.341955', 'currency': 'USD'}","{'amount': '2.412918', 'currency': 'USD'}","{'amount': '185559.124598', 'currency': 'USD'}"
8,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2243,"[601352, 603014, 604453]",Y5EkgwAAAACOV4AEeERJl-yievH_HBssTrKMtvBlwwqFdF...,88806,{'formatted_range': '₱10K+'},Jp Olimar🇵🇭,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jpongs_,NaN,NaN,NaN


In [106]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,video_gmv,live_gmv
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601352, 601152]",I2IWFwAAAACOV4AEeERJl-yievH_HBssOVaC2_feBRCR7I...,895,{'formatted_range': '₱100-₱1K'},Jpongs_,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",jpolimar2nd,NaN,NaN,NaN
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2243,"[601352, 603014, 604453]",Y5EkgwAAAACOV4AEeERJl-yievH_HBssTrKMtvBlwwqFdF...,88806,{'formatted_range': '₱10K+'},Jp Olimar🇵🇭,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jpongs_,NaN,NaN,NaN
2,{'url': 'https://p19-common-sign.tiktokcdn.com...,0,1385,"[600942, 801928, 602284]",OjX16gAAAACOV4AEeERJl-yievH_HBssLMogjZiSaMeXxo...,27338,{'formatted_range': '₱10K+'},Karen Anonas,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",karenanonas,NaN,NaN,NaN
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601739, 600001, 604453]",S7jHnQAAAACOV4AEeERJl-yievH_HBsspmDYP-pIAVrUlv...,31734,{'formatted_range': '₱10K+'},Almira Carriaga,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",almiracarriaga,NaN,NaN,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",9Ovn_wAAAACOV4AEeERJl-yievH_HBssS7BIL5paEZ6qYL...,531,{'formatted_range': '₱10K+'},Maria,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",user5844826884387,NaN,NaN,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",3MxrrwAAAACOV4AEeERJl-yievH_HBssPMIZu50b8vQOzm...,12,"{'currency': 'USD', 'formatted_range': '₱10K+'...",iflyad03,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",iflyad03,"{'amount': '5637882.296699', 'currency': 'USD'}",NaN,NaN
6,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 605196]",T51HRwAAAACOV4AEeERJl-yievH_HBsscpim0sGFIiscz9...,1,{'formatted_range': '₱10K+'},ShopAtHomePH,PH,"{'age_ranges': ['AGE_RANGE_35_44'], 'major_gen...",shopathomeph,NaN,NaN,NaN
7,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601739, 605196, 601450]",Bk01NQAAAACOV4AEeERJl-yievH_HBssjYBT2NCiw6gm5a...,787,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Nicolas Pagquil,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",nicolas.pagquil,"{'amount': '28293.612369', 'currency': 'USD'}",NaN,NaN
8,{'url': 'https://p16-common-sign.tiktokcdn.com...,102597,0,"[601739, 601755]",jSATJgAAAACOV4AEeERJl-yievH_HBssugnh2geIdZYbIX...,114602,{'formatted_range': '₱10K+'},Hari ng Chamba,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",haringchamba,NaN,NaN,NaN
9,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,263,"[601739, 604968, 605196]",4WFi3gAAAACOV4AEeERJl-yievH_HBss5966XF9PQoqdpS...,20694,"{'currency': 'USD', 'formatted_range': '₱10K+'...",Egi finds,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",egii_tv_06,"{'amount': '22370.600599', 'currency': 'USD'}","{'amount': '22370.600599', 'currency': 'USD'}",NaN


In [87]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv,gmv_range,nickname,selection_region,top_follower_demographics,username,video_gmv,live_gmv
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,3629,"[600001, 700437, 601739]",0Ktl2wAAAACOV4AEeERJl-yievH_HBss98ZZMcc5NxLGOJ...,127171,"{'amount': '199035.500849', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",david sy,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",david.sy25,"{'amount': '198947.459305', 'currency': 'USD'}",NaN
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,1854,"[601352, 824328, 601152]",OlDpxwAAAACOV4AEeERJl-yievH_HBssyJrMiLFJf1B0bD...,26274,"{'amount': '105158.110182', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",Fernando's Finds,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",majuuuuuuri,"{'amount': '103080.691459', 'currency': 'USD'}",NaN
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,9792,39,"[601152, 605248, 824328]",vuvQSgAAAACOV4AEeERJl-yievH_HBssTBNmqgnKaxC9GP...,31628,"{'amount': '26321.024049', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",Kikayshop.ph,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",kikayshop.ph21,"{'amount': '536.07245', 'currency': 'USD'}","{'amount': '25773.195083', 'currency': 'USD'}"
3,{'url': 'https://p19-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",9Ovn_wAAAACOV4AEeERJl-yievH_HBssS7BIL5paEZ6qYL...,531,NaN,{'formatted_range': '₱10K+'},Maria,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",user5844826884387,NaN,NaN
4,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 600001]",3MxrrwAAAACOV4AEeERJl-yievH_HBssPMIZu50b8vQOzm...,12,"{'amount': '5637882.296699', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",iflyad03,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",iflyad03,NaN,NaN
5,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601450, 601739, 605196]",T51HRwAAAACOV4AEeERJl-yievH_HBsscpim0sGFIiscz9...,1,NaN,{'formatted_range': '₱10K+'},ShopAtHomePH,PH,"{'age_ranges': ['AGE_RANGE_35_44'], 'major_gen...",shopathomeph,NaN,NaN
6,{'url': 'https://p16-common-sign.tiktokcdn.com...,3482,645,"[601152, 824328, 603014]",rvPU_wAAAACOV4AEeERJl-yievH_HBssu77XDSFsZ1iW_V...,61375,"{'amount': '24627.49385', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",Tita Life 🌸,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",tipid.tita,"{'amount': '12495.906165', 'currency': 'USD'}","{'amount': '12023.621935', 'currency': 'USD'}"
7,{'url': 'https://p16-common-sign.tiktokcdn.com...,6950,1464,"[601152, 605248, 601352]",3z_RngAAAACOV4AEeERJl-yievH_HBsslIUL-1c7pIG4ZK...,143433,"{'amount': '44726.476945', 'currency': 'USD'}","{'currency': 'USD', 'formatted_range': '₱10K+'...",Miss CL,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",msclstudio_3448,"{'amount': '36987.043833', 'currency': 'USD'}","{'amount': '7683.747675', 'currency': 'USD'}"
8,{'url': 'https://p16-common-sign.tiktokcdn.com...,1407,394,"[601152, 700437, 601739]",BrkGhwAAAACOV4AEeERJl-yievH_HBssTkYOrDle_jJAcY...,4430,NaN,{'formatted_range': '₱10K+'},nanaydump🥑,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",amoireelise,NaN,NaN
9,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,1882,"[601152, 600154, 824328]",yXrqVQAAAACOV4AEeERJl-yievH_HBssu5qAeG-D6YTxYx...,83879,NaN,{'formatted_range': '₱10K+'},. ݁˖ ʚjovjeanɞ . ݁˖,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jovjean,NaN,NaN


In [66]:
pd.DataFrame(response.json()['data']['creators'])

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username
0,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[601352, 601152]",I2IWFwAAAACOV4AEeERJl-yievH_HBssOVaC2_feBRCR7I...,895,{'formatted_range': '₱100-₱1K'},Jpongs_,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",jpolimar2nd
1,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[700645, 601739, 602284]",FJXiWgAAAACOV4AEeERJl-yievH_HBss6kZoZmHqG7TSuA...,1923,{'formatted_range': '₱0-₱100'},jomaryeedump,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedump_2000
2,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,0,"[602118, 824584, 601739]",L7PcYwAAAACOV4AEeERJl-yievH_HBsshmh4OB3xdmSeP7...,1004,{'formatted_range': '₱0-₱100'},yuni okayde😊,PH,"{'age_ranges': ['AGE_RANGE_25_34', 'AGE_RANGE_...",jomaryeedumpz
3,{'url': 'https://p16-common-sign.tiktokcdn.com...,0,2243,"[601352, 603014, 604453]",Y5EkgwAAAACOV4AEeERJl-yievH_HBssTrKMtvBlwwqFdF...,88806,{'formatted_range': '₱10K+'},Jp Olimar🇵🇭,PH,"{'age_ranges': ['AGE_RANGE_18_24'], 'major_gen...",jpongs_
4,{'url': 'https://p19-common-sign.tiktokcdn.com...,0,1385,"[600942, 801928, 602284]",OjX16gAAAACOV4AEeERJl-yievH_HBssLMogjZiSaMeXxo...,27338,{'formatted_range': '₱10K+'},Karen Anonas,PH,"{'age_ranges': ['AGE_RANGE_25_34'], 'major_gen...",karenanonas


In [58]:
response.json()

{'code': 0,
 'data': {'creators': [{'avatar': {'url': 'https://p16-common-sign.tiktokcdn.com/tos-alisg-avt-0068/c4cca57c29540180af9d3efd4fffd73f~tplv-tiktokx-cropcenter:720:720.webp?dr=14579&refresh_token=fa7d2588&x-expires=1784034000&x-signature=hbyGzPtjQ4%2BeKQc5r48udYPd8%2FM%3D&t=4d5b0474&ps=13740610&shp=a5d48078&shcp=39dffb78&idc=my2'},
    'avg_ec_live_uv': 0,
    'avg_ec_video_view_count': 0,
    'category_ids': ['601352', '601152'],
    'creator_open_id': 'I2IWFwAAAACOV4AEeERJl-yievH_HBssOVaC2_feBRCR7IlgScbk9w',
    'follower_count': 895,
    'gmv_range': {'formatted_range': '₱100-₱1K'},
    'nickname': 'Jpongs_',
    'selection_region': 'PH',
    'top_follower_demographics': {'age_ranges': ['AGE_RANGE_25_34'],
     'major_gender': {'gender': 'FEMALE', 'percentage': 4217}},
    'username': 'jpolimar2nd'},
   {'avatar': {'url': 'https://p16-common-sign.tiktokcdn.com/tos-maliva-avt-0068/d75bb4500cefbbe712fe4ae267d70cef~tplv-tiktokx-cropcenter:720:720.webp?dr=14579&refresh_token=6a